In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
import optuna
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import accuracy_score
from optuna.visualization import plot_terminator_improvement
from optuna.terminator import report_cross_validation_scores
import json
import matplotlib.pyplot as plt
from tabulate import tabulate

In [2]:
X_train= pd.read_csv('/kaggle/input/datasets/prahazra/employee-attrition-processed/X_train_final.csv')
x_test= pd.read_csv('/kaggle/input/datasets/prahazra/employee-attrition-processed/X_test_final.csv')

Y_train= pd.read_csv('/kaggle/input/datasets/prahazra/employee-attrition-processed/y_train_final.csv').squeeze()
y_test= pd.read_csv('/kaggle/input/datasets/prahazra/employee-attrition-processed/y_test_final.csv')

In [3]:
print(Y_train.value_counts())
print(len(X_train))

Attrition
0    986
1    986
Name: count, dtype: int64
1972


In [4]:
x_test.drop(columns=['Unnamed: 0'], inplace= True)
y_test.drop(columns=['Unnamed: 0'], inplace= True)

In [5]:
if hasattr(y_test, 'squeeze'):
    y_test = y_test.squeeze()

In [6]:
y_test

0      0
1      0
2      1
3      1
4      0
      ..
289    0
290    0
291    0
292    0
293    0
Name: Attrition, Length: 294, dtype: int64

# Performance Before Optimization

In [7]:
LR= LogisticRegression(random_state= 0)
RF= RandomForestClassifier(random_state = 0)
GB= GradientBoostingClassifier(random_state= 0)
XGB= XGBClassifier(random_state= 0)

In [8]:
LR.fit(X_train, Y_train)
RF.fit(X_train, Y_train)
GB.fit(X_train, Y_train)
XGB.fit(X_train, Y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [9]:
Models= {
    "Logistic Regression": LR,
    "Random Forest": RF,
    "Gradient Boosting": GB,
    "XGBoost": XGB
}

performance_BO= []
for name, model in Models.items():
    y_pred= model.predict(x_test)
    accuracy = accuracy_score(y_pred, y_test)
    performance_BO.append([name, f"{accuracy * 100:.2f}%"])

headers= ['Model', 'Test Accuracy']
print(tabulate(performance_BO, headers= headers, tablefmt= 'grid'))

+---------------------+-----------------+
| Model               | Test Accuracy   |
+=====================+=================+
| Logistic Regression | 75.51%          |
+---------------------+-----------------+
| Random Forest       | 85.03%          |
+---------------------+-----------------+
| Gradient Boosting   | 87.76%          |
+---------------------+-----------------+
| XGBoost             | 86.73%          |
+---------------------+-----------------+


# Model Optimization using Optuna

In [10]:
sampler= optuna.samplers.TPESampler(seed =0)

In [11]:
def objective_LR(trial):
    params= {
        'penalty' : trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet']),
        'C' : trial.suggest_float('C', 1e-5, 10.0, log=True),
        'tol' : trial.suggest_float('tol', 1e-6, 1e-1, log= True),
        'solver' : trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']),
        'max_iter' : trial.suggest_int('max_iter', 50 , 200)
    }
    if params['penalty'] == 'elasticnet':
        params['l1_ratio'] = trial.suggest_float('l1_ratio', 0 ,1)
    
    try:
        model= LogisticRegression(**params, random_state = 0, n_jobs = -1)
        scores = cross_val_score(model, X_train, Y_train, 
                                 cv=KFold(n_splits=5, shuffle=True, random_state= 0))
        
        report_cross_validation_scores(trial, scores)
        return np.mean(scores)
    except Exception as e:
        raise optuna.TrialPruned(f"Pruned due to runtime error: {e}")

In [12]:
study_LR = optuna.create_study(direction="maximize", study_name= "Optimize_LR", sampler= sampler)
study_LR.optimize(objective_LR, n_trials=50, show_progress_bar=True)
fig_LR= plot_terminator_improvement(study_LR, plot_error = True)
fig_LR.show()

[I 2026-06-05 11:28:24,583] A new study created in memory with name: Optimize_LR


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-05 11:28:30,080] Trial 0 finished with value: 0.7941309516160123 and parameters: {'penalty': 'l2', 'C': 0.01859084363016962, 'tol': 0.0001313028028065861, 'solver': 'newton-cholesky', 'max_iter': 129}. Best is trial 0 with value: 0.7941309516160123.
[I 2026-06-05 11:28:30,175] Trial 1 finished with value: 0.6668701407183705 and parameters: {'penalty': 'l2', 'C': 3.332543279005119e-05, 'tol': 1.2620948285169311e-06, 'solver': 'newton-cholesky', 'max_iter': 167}. Best is trial 0 with value: 0.7941309516160123.
[I 2026-06-05 11:28:30,337] Trial 2 finished with value: 0.8169440339266207 and parameters: {'penalty': 'l2', 'C': 4.656005689076002, 'tol': 0.0004066695065393846, 'solver': 'newton-cg', 'max_iter': 143}. Best is trial 2 with value: 0.8169440339266207.
[I 2026-06-05 11:28:30,360] Trial 3 pruned. Pruned due to runtime error: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  

[I 2026-06-05 11:28:31,025] Trial 6 finished with value: 0.7332686500032127 and parameters: {'penalty': 'l2', 'C': 0.0004975557797102343, 'tol': 3.9900910508086484e-06, 'solver': 'saga', 'max_iter': 135}. Best is trial 2 with value: 0.8169440339266207.
[I 2026-06-05 11:28:31,116] Trial 7 finished with value: 0.802742401850543 and parameters: {'penalty': 'l2', 'C': 0.028554790180743944, 'tol': 0.04430788173937543, 'solver': 'newton-cholesky', 'max_iter': 138}. Best is trial 2 with value: 0.8169440339266207.
[I 2026-06-05 11:28:31,217] Trial 8 finished with value: 0.8149071515774594 and parameters: {'penalty': 'l2', 'C': 0.11665388871103255, 'tol': 2.2389266508862164e-05, 'solver': 'liblinear', 'max_iter': 83}. Best is trial 2 with value: 0.8169440339266207.
[I 2026-06-05 11:28:31,234] Trial 9 pruned. Pruned due to runtime error: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more detai

...

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[I 2026-06-05 11:28:38,522] Trial 46 finished with value: 0.8128856904195849 and parameters: {'penalty': 'l2', 'C': 3.049677082330909, 'tol': 4.5247902169109474e-05, 'solver': 'saga', 'max_iter': 126}. Best is trial 16 with value: 0.8179618325515646.
[I 2026-06-05 11:28:38,546] Trial 47 pruned. Pruned due to runtime error: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  

[I 2026-06-05 11:28:38,796] Trial 48 finished with value: 0.8144046777613569 and parameters: {'penalty': 'l2', 'C': 1.4399095471897894, 'tol': 1.0231779319978908e-05, 'solver': 'lbfgs', 'max_iter': 143}. Best is trial 16 with value: 0.8179618325515646.
[I 2026-06-05 11:28:38,896] Trial 49 finished with value: 0.810855233566793 and parameters: {'penalty': 'l2', 'C': 0.9797235421664999, 'tol': 0.006192218483854524, 'solver': 'liblinear', 'max_iter': 163}. Best is trial 16 with value: 0.8179618325515646.


100%|██████████| 50/50 [00:08<00:00,  5.56it/s]


In [13]:
def objective_RF(trial):
    params= {
        'n_estimators' : trial.suggest_int('n_estimators', 100, 300),
        'criterion' : trial.suggest_categorical('criterion', ['gini','entropy','log_loss']),
        'max_depth' : trial.suggest_int('max_depth', 5 ,10),
        'min_samples_split' : trial.suggest_int('min_samples_split', 10, 40),
        'bootstrap' : trial.suggest_categorical('bootstrap', [True, False]),
        'class_weight': trial.suggest_categorical('class_weight', ['balanced', 'balanced_subsample', None])
    }
    max_features_option = trial.suggest_categorical('max_features_option', ['sqrt', 'log2', None, 'float']) # Corrected parameter name
    if max_features_option == 'float':
        params['max_features'] = trial.suggest_float('max_features', 0.1, 1.0)
    else:
        params['max_features'] = max_features_option
    if params['bootstrap'] == True:
        params['max_samples']= trial.suggest_float('max_samples', 0.5, 0.9)

    try:
        model= RandomForestClassifier(**params, random_state = 0, n_jobs = -1)
        scores = cross_val_score(model, X_train, Y_train, 
                                 cv=KFold(n_splits=5, shuffle=True, random_state= 0))
        
        report_cross_validation_scores(trial, scores)
        return np.mean(scores)
    except Exception as e:
        raise optuna.TrialPruned(f"Pruned due to runtime error: {e}")

In [14]:
study_RF = optuna.create_study(direction="maximize", study_name= "Optimize_RF", sampler= sampler)
study_RF.optimize(objective_RF, n_trials=50, show_progress_bar=True)
fig_RF= plot_terminator_improvement(study_RF, plot_error = True)
fig_RF.show()

[I 2026-06-05 11:28:49,830] A new study created in memory with name: Optimize_RF


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-05 11:28:59,766] Trial 0 finished with value: 0.9173436998008097 and parameters: {'n_estimators': 200, 'criterion': 'gini', 'max_depth': 8, 'min_samples_split': 10, 'bootstrap': False, 'class_weight': 'balanced_subsample', 'max_features_option': 'float', 'max_features': 0.6168927239646209}. Best is trial 0 with value: 0.9173436998008097.
[I 2026-06-05 11:29:03,137] Trial 1 finished with value: 0.9143082953158131 and parameters: {'n_estimators': 231, 'criterion': 'log_loss', 'max_depth': 7, 'min_samples_split': 23, 'bootstrap': True, 'class_weight': None, 'max_features_option': 'log2', 'max_samples': 0.5649971738705499}. Best is trial 0 with value: 0.9173436998008097.
[I 2026-06-05 11:29:07,282] Trial 2 finished with value: 0.9279997429801453 and parameters: {'n_estimators': 223, 'criterion': 'entropy', 'max_depth': 8, 'min_samples_split': 22, 'bootstrap': False, 'class_weight': None, 'max_features_option': 'sqrt'}. Best is trial 2 with value: 0.9279997429801453.
[I 2026-06-0

/tmp/ipykernel_16/2767116058.py:3: ExperimentalWarning:

optuna.visualization._terminator_improvement.plot_terminator_improvement is experimental (supported from v3.2.0). The interface can change in the future.

/usr/local/lib/python3.12/dist-packages/optuna/visualization/_terminator_improvement.py:93: ExperimentalWarning:

RegretBoundEvaluator is experimental (supported from v3.2.0). The interface can change in the future.

/usr/local/lib/python3.12/dist-packages/optuna/visualization/_terminator_improvement.py:98: ExperimentalWarning:

CrossValidationErrorEvaluator is experimental (supported from v3.2.0). The interface can change in the future.



[I 2026-06-05 11:33:17,451] Trial 49 finished with value: 0.9330784553106728 and parameters: {'n_estimators': 291, 'criterion': 'log_loss', 'max_depth': 9, 'min_samples_split': 15, 'bootstrap': False, 'class_weight': None, 'max_features_option': 'log2'}. Best is trial 29 with value: 0.9391595450748571.


100%|██████████| 50/50 [00:05<00:00,  8.63it/s]


In [15]:
def objective_GB(trial):
    params = {
        'loss': trial.suggest_categorical('loss', ['log_loss', 'exponential']),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'criterion': trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error']),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_impurity_decrease': trial.suggest_float('min_impurity_decrease', 0.0, 0.5),
        'max_depth': trial.suggest_int('max_depth', 1, 20),
        'random_state': 0
    }
    max_features_option = trial.suggest_categorical('max_features_option', ['sqrt', 'log2', None, 'float']) 
    if max_features_option == 'float':
        params['max_features'] = trial.suggest_float('max_features_gb_float', 0.1, 1.0)
    else:
        params['max_features'] = max_features_option

    try:
        model = GradientBoostingClassifier(**params)
        scores = cross_val_score(model, X_train, Y_train, cv=KFold(n_splits=5, shuffle=True,random_state= 0))
        report_cross_validation_scores(trial, scores)
        return np.mean(scores)
    except Exception as e:
        raise optuna.TrialPruned(f"Pruned due to runtime error: {e}")

In [16]:
study_GB = optuna.create_study(direction="maximize", study_name= "Optimize_GB", sampler = sampler)
study_GB.optimize(objective_GB, n_trials=50, show_progress_bar=True)
fig_GB= plot_terminator_improvement(study_GB, plot_error = True)
fig_GB.show()

[I 2026-06-05 11:33:23,377] A new study created in memory with name: Optimize_GB


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-05 11:33:25,452] Trial 0 finished with value: 0.9280035982779669 and parameters: {'loss': 'log_loss', 'learning_rate': 0.1956272447321156, 'n_estimators': 182, 'criterion': 'friedman_mse', 'min_samples_split': 2, 'min_impurity_decrease': 0.17361675896610979, 'max_depth': 3, 'max_features_option': 'sqrt'}. Best is trial 0 with value: 0.9280035982779669.
[I 2026-06-05 11:33:27,184] Trial 1 finished with value: 0.9284970763991518 and parameters: {'loss': 'log_loss', 'learning_rate': 0.1638202474445442, 'n_estimators': 135, 'criterion': 'friedman_mse', 'min_samples_split': 3, 'min_impurity_decrease': 0.43109575871084166, 'max_depth': 20, 'max_features_option': 'sqrt'}. Best is trial 1 with value: 0.9284970763991518.
[I 2026-06-05 11:33:30,813] Trial 2 finished with value: 0.47312985928162954 and parameters: {'loss': 'exponential', 'learning_rate': 0.02203119160109112, 'n_estimators': 109, 'criterion': 'squared_error', 'min_samples_split': 2, 'min_impurity_decrease': 0.3852903742

/tmp/ipykernel_16/3313013327.py:3: ExperimentalWarning:

optuna.visualization._terminator_improvement.plot_terminator_improvement is experimental (supported from v3.2.0). The interface can change in the future.

/usr/local/lib/python3.12/dist-packages/optuna/visualization/_terminator_improvement.py:93: ExperimentalWarning:

RegretBoundEvaluator is experimental (supported from v3.2.0). The interface can change in the future.

/usr/local/lib/python3.12/dist-packages/optuna/visualization/_terminator_improvement.py:98: ExperimentalWarning:

CrossValidationErrorEvaluator is experimental (supported from v3.2.0). The interface can change in the future.



[I 2026-06-05 11:40:56,442] Trial 49 finished with value: 0.9427064190708732 and parameters: {'loss': 'exponential', 'learning_rate': 0.033014270854650564, 'n_estimators': 171, 'criterion': 'friedman_mse', 'min_samples_split': 20, 'min_impurity_decrease': 0.029597218676590743, 'max_depth': 11, 'max_features_option': 'sqrt'}. Best is trial 44 with value: 0.9482888903167769.


100%|██████████| 50/50 [00:05<00:00,  8.84it/s]


In [17]:
def objective_XGB(trial):
    params = {
      'n_estimators': trial.suggest_int('n_estimators', 50, 500),
      'max_depth': trial.suggest_int('max_depth', 1, 20),
      'grow_policy': trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide']), 
      'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
      'booster': trial.suggest_categorical('booster', ['gbtree', 'gblinear', 'dart']),
      'gamma': trial.suggest_float('gamma', 0.0, 0.5),
      'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
      'use_label_encoder': False, 
      'eval_metric': 'logloss', 
      'random_state': 0
    }
    
    try:
      model = XGBClassifier(**params, n_jobs= -1)
      scores = cross_val_score(model, X_train, Y_train, cv=KFold(n_splits=5, shuffle=True,random_state= 0))
      report_cross_validation_scores(trial, scores)
    
      return np.mean(scores)
    
    except Exception as e:
      raise optuna.TrialPruned(f"Pruned due to runtime error: {e}")

In [18]:
study_XGB = optuna.create_study(direction="maximize", study_name= "Optimize_XGB", sampler= sampler)
study_XGB.optimize(objective_XGB, n_trials=50, show_progress_bar=True)
fig_XGB= plot_terminator_improvement(study_XGB, plot_error = True)
fig_XGB.show()

[I 2026-06-05 11:41:02,250] A new study created in memory with name: Optimize_XGB


  0%|          | 0/50 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[11:41:02] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[11:41:02] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[11:41:02] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[11:41:03] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[11:41:03] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.




[I 2026-06-05 11:41:03,740] Trial 0 finished with value: 0.9026357386108076 and parameters: {'n_estimators': 57, 'max_depth': 9, 'grow_policy': 'lossguide', 'learning_rate': 0.02121687847073361, 'booster': 'gbtree', 'gamma': 0.05774214856937404, 'min_child_weight': 7}. Best is trial 0 with value: 0.9026357386108076.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[11:41:03] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "gamma", "grow_policy", "max_depth", "min_child_weight", "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[11:41:04] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "gamma", "grow_policy", "max_depth", "min_child_weight", "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[11:41:04] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "gamma", "grow_policy", "max_depth", "min_child_weight", "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[11:41:04] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "gamma", "grow_policy", "max_depth", "min_child_weight", "use_label_encoder" } are not used.




...

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[11:54:08] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "gamma", "grow_policy", "max_depth", "min_child_weight", "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[11:54:08] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "gamma", "grow_policy", "max_depth", "min_child_weight", "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[11:54:09] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "gamma", "grow_policy", "max_depth", "min_child_weight", "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[11:54:09] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "gamma", "grow_policy", "max_depth", "min_child_weight", "use_label_encoder" } are not used.




[I 2026-06-05 11:54:10,018] Trial 48 finished with value: 0.8184643063676669 and parameters: {'n_estimators': 499, 'max_depth': 15, 'grow_policy': 'lossguide', 'learning_rate': 0.173357820716154, 'booster': 'gblinear', 'gamma': 0.3685791230996379, 'min_child_weight': 2}. Best is trial 33 with value: 0.9422052303540449.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[11:54:10] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[11:54:11] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[11:54:13] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[11:54:15] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[11:54:17] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.


/tmp/ipykernel_16/2259274934.py:3: 

[I 2026-06-05 11:54:19,383] Trial 49 finished with value: 0.9315491871747094 and parameters: {'n_estimators': 463, 'max_depth': 14, 'grow_policy': 'lossguide', 'learning_rate': 0.018438371690404673, 'booster': 'gbtree', 'gamma': 0.22368913057588172, 'min_child_weight': 4}. Best is trial 33 with value: 0.9422052303540449.


100%|██████████| 50/50 [00:05<00:00,  8.83it/s]


In [19]:
Model_names= ['Logistic Regression', 'Random Forest', 'Gradient Boosting', 'XGBoost']
Model_scores= [study_LR.best_trial.value, study_RF.best_trial.value, study_GB.best_trial.value, study_XGB.best_trial.value]
Model_params= [study_LR.best_trial.params, study_RF.best_trial.params, study_GB.best_trial.params, study_XGB.best_trial.params]
performance= pd.DataFrame({'Model': Model_names, 'Score': Model_scores, 'Parameters': Model_params})
performance[['Model', 'Score']]

,Model,Score
0,Logistic Regression,0.817962
1,Random Forest,0.939160
2,Gradient Boosting,0.948289
3,XGBoost,0.942205


In [20]:
def save_params(prefix_name, params_dict):
    file_name= f"{prefix_name}.json"
    with open(file_name, "w") as f:
        json.dump(params_dict, f, indent= 4)
    print(f"Successfully saved to: {file_name}")

In [21]:
LR_params= study_LR.best_params
save_params("LR_params", LR_params)
RF_params= study_RF.best_params
save_params('RF_params', RF_params)
GB_params= study_GB.best_params
save_params("GB_params", GB_params)
XGB_params= study_XGB.best_params
save_params("XGB_params", XGB_params)

Successfully saved to: LR_params.json
Successfully saved to: RF_params.json
Successfully saved to: GB_params.json
Successfully saved to: XGB_params.json
